### Build-in Types

<img src="../Pics/data_types.png" width="300">

`int.__sub__` is a **built-in method descriptor**, not a normal Python function.  
When it runs, it ends up in **CPython’s C code** for integer subtraction.

The `-` operator bytecode also ends up calling that **same C subtraction logic**,  
just via a faster **direct slot** route instead of doing attribute lookup for `"__sub__"`.

So for `int`:

- `2 - 1` → subtraction opcode → `int`’s subtraction slot → C integer-subtract  
- `(2).__sub__(1)` → normal attribute lookup → built-in method call → C integer-subtract  

**What happens for `h * 10`**

- The `*` opcode triggers CPython’s C-level numeric **multiplication dispatch**.
- That C dispatch checks whether the **left operand’s type** (`Hello`) provides a multiply handler:
  - internally, a numeric “slot” like `nb_multiply`, derived from `Hello.__mul__`.
- If present → it calls it (conceptually):

  - `Hello.__mul__(h, 10)`  
  - i.e. `type(h).__mul__(h, 10)`

- If there’s **no handler**, or it returns **`NotImplemented`** → CPython tries the **right operand’s reflected method**:

  - `int.__rmul__(10, h)`

- If neither side can handle it → raises:

  - `TypeError: unsupported operand type(s) for *: ...`


**How this differs from `2 - 1`**

1) **For `2 - 1`, there’s no Python-level method lookup in practice**

- For `int`, subtraction is implemented in C via the number slot:

  - `PyLong_Type.tp_as_number->nb_subtract`

- So CPython just runs the C integer-subtraction routine and returns a new `int`.
- That usually means:
  - no instance dictionary lookup
  - no calling a Python function object
  - no executing user bytecode

2) **For `h * 10`, your `Hello.__mul__` is Python code**

- CPython still uses the same slot mechanism, but that slot points to a **wrapper** that calls your Python method.
- That typically involves:
  - creating a Python call frame
  - executing your method’s bytecode (`__mul__`)
  - returning the result

### Data Structures

<img src="../Pics/data_struc.png" width="800">

In [4]:
job_skills = ['python', 'excel', 'sql', 'looker']

skill_concerned, *skill_dont_care = job_skills
print(skill_concerned)
print(skill_dont_care)

python
['excel', 'sql', 'looker']


In [ ]:
# Fastest lookups in array
# 1 lots of membership checkes
s = set(arr)          # one-time cost O(n)
found = target in s   # fast --> O(1)

# 2 one-off check, small/medium list
found = target in arr

# 3 sorted list --> binary search O(log n)
import bisect
i = bisect.bisect_left(arr, target)
found = i < len(arr) and arr[i] == target


In [ ]:
# Usage of Set as a way to get unique values
skill_list = ['python', 'sql', 'statistics', 'tableau', 'python', 'sql', 'statistics', 'tableau']
set(skill_list)

{'python', 'sql', 'statistics', 'tableau'}

### Bitwise Operators

<img src="../Pics/bitwise_oper.png" width="300">

In [9]:
bin(42)
bin(34 << 8)

'0b10001000000000'

### Exercise 1

Filter jobs to apply for based on a list of skills

In [26]:
# Define data science job roles and required skills
job_roles = [
    {'role': 'Data Analyst', 'skills': ['Python', 'SQL', 'Excel']},
    {'role': 'Data Scientist', 'skills': ['Python', 'R', 'Machine Learning', 'Deep Learning']},
    {'role': 'Machine Learning Engineer', 'skills': ['Python', 'TensorFlow', 'PyTorch', 'Scikit-Learn']},
    {'role': 'Data Engineer', 'skills': ['Python', 'Apache Spark', 'Hadoop', 'SQL']},
    {'role': 'Business Intelligence Analyst', 'skills': ['Python', 'SQL', 'Tableau', 'Power BI', 'Excel']},
    {'role': 'Quantitative Analyst', 'skills': ['R', 'Python', 'MATLAB', 'Statistics']},
    {'role': 'Operations Analyst', 'skills': ['Python', 'SQL', 'Data Visualization', 'Process Improvement']},
    {'role': 'Database Administrator', 'skills': ['SQL', 'Oracle', 'MySQL', 'Database Management']},
    {'role': 'AI Engineer', 'skills': ['Python', 'TensorFlow', 'PyTorch', 'Computer Vision']},
    {'role': 'Statistician', 'skills': ['R', 'SAS', 'Python', 'Statistical Modeling']}
]
# My skills
my_skills = ['Python', 'SQL', 'Excel']

# 1 - Fastest and most efficent
my_skills_set = set(my_skills)
job_roles_sets = [{"role": jr["role"], "skills_set": set(jr["skills"])} for jr in job_roles]

for jr in job_roles_sets:
    if my_skills_set.issubset(jr['skills_set']):  # same as <=
        print(jr['role'])

print()

# 2 Concise
for job in job_roles:
    # Check if all required skills are in my_skills
    if all(skill in job['skills'] for skill in my_skills):
        print(job['role'])

Data Analyst
Business Intelligence Analyst

Data Analyst
Business Intelligence Analyst


### Types of Functions

<img src="../Pics/types_fun.png" width="400">

In [ ]:
import types

# __builtins__ is a dunder name
# built-in namespace reference which points to
# the builtins module or its dict (a mapping of built-in names -> objs)

# list the built-in functions
print([func for func in dir(__builtins__) if isinstance(getattr(__builtins__, func), types.BuiltinFunctionType)])

['__build_class__', '__import__', 'abs', 'aiter', 'all', 'anext', 'any', 'ascii', 'bin', 'breakpoint', 'callable', 'chr', 'compile', 'delattr', 'dir', 'divmod', 'eval', 'exec', 'format', 'getattr', 'globals', 'hasattr', 'hash', 'hex', 'id', 'isinstance', 'issubclass', 'iter', 'len', 'locals', 'max', 'min', 'next', 'oct', 'open', 'ord', 'pow', 'print', 'repr', 'round', 'setattr', 'sorted', 'sum', 'vars']


**Namespace** = “where names are stored” (like a dict mapping names to values).

**Dunder names** = “names Python treats specially.”

Some dunder names store normal values, some store functions/methods, and a few store mappings/modules (which can act like namespaces).

1) Special methods (a.k.a. magic methods)
Examples: `__len__`, `__iter__`, `__add__`, `__init__`, etc.

- These are **callable hooks**.
- Python (and built-ins like `len`, operators like `+`, iteration, context managers, etc.) will **invoke them implicitly** to implement language features.
2) Dunder attributes / metadata
Examples: `__dict__`, `__class__`, `__mro__`, `__annotations__`, `__slots__`, etc.

- These are usually **data/metadata attributes**, not “hooks”.
- They describe or store information about an object/class (e.g., its attribute storage, its type, method resolution order, type hints, slot layout).


In [29]:
# lambda <arguments>: <expression>

mul_two = lambda x: x*2
mul_two(2)

4

In [30]:
(lambda x, y: x*2 + y)(3, 7)

13

**When they’re useful**
- **Short callbacks** (especially with `sorted`, `min`, `max`)
```python
words = ["pear", "apple", "banana"]
words_sorted = sorted(words, key=lambda s: len(s))
```
- **Transforming/filtering** with `map` / `filter` (though comprehensions are often clearer)
```python
nums = [1, 2, 3, 4]
evens = list(filter(lambda n: n % 2 == 0, nums))
squares = list(map(lambda n: n * n, nums))
```
- **“Key” functions** based on part of an item
```python
pairs = [(1, "b"), (3, "a"), (2, "c")]
sorted_pairs = sorted(pairs, key=lambda t: t[1])
```

In [38]:
salary_list = [1100000, 200000, 150000, 120000, 80000, 750000]

# A Calculate total_salary of base_salary * (1 + bonus_rate)
def calculate_salary (base_salary, bonus_rate=.1):
    return base_salary * (1 + bonus_rate)
total_salary_list = [calculate_salary(salary) for salary in salary_list]

print(total_salary_list)

total_salary_list_lambda = [(lambda x: x*1.1)(salary) for salary in salary_list]

print(total_salary_list_lambda)

[1210000.0, 220000.00000000003, 165000.0, 132000.0, 88000.0, 825000.0000000001]
[1210000.0, 220000.00000000003, 165000.0, 132000.0, 88000.0, 825000.0000000001]


In [2]:
# Calculate Average Salary
salaries = [95000, 120000, 105000, 90000, 130000]

average_salary = lambda salary_list: sum(salary_list)/len(salary_list)
average = average_salary(salaries)
average

108000.0

In [10]:
# Filter Job Titles
job_titles = ['Data Scientist', 'Data Engineer', 'Machine Learning Engineer', 'Data Analyst']

list(filter(lambda title: 'Data' in title, job_titles))

['Data Scientist', 'Data Engineer', 'Data Analyst']

In [12]:
# Filter Remote Python Jobs

job_postings = [
    {'title': 'Data Scientist', 'skills': ['Python', 'SQL'], 'remote': True},
    {'title': 'Data Analyst', 'skills': ['Excel', 'SQL'], 'remote': False},
    {'title': 'Machine Learning Engineer', 'skills': ['Python', 'TensorFlow'], 'remote': True},
    {'title': 'Software Developer', 'skills': ['Java', 'C++'], 'remote': True}
]

list(filter(lambda posting: 'Python' in posting['skills'] and posting['remote'], job_postings))

[{'title': 'Data Scientist', 'skills': ['Python', 'SQL'], 'remote': True},
 {'title': 'Machine Learning Engineer',
  'skills': ['Python', 'TensorFlow'],
  'remote': True}]

### Python Standard Library

In [13]:
from statistics import mean, mode, median

salaries = [95000, 120000, 105000, 90000, 130000]
print(mean(salaries))
print(mode(salaries))
print(median(salaries))

108000
95000
105000


### Exercise 2 (Data Cleanup)

In [ ]:
from datetime import datetime
import ast

data_science_jobs = [
    {'job_title': 'Data Scientist', 'job_skills': "['Python', 'SQL', 'Machine Learning']", 'job_date': '2023-05-12'},
    {'job_title': 'Machine Learning Engineer', 'job_skills': "['Python', 'TensorFlow', 'Deep Learning']", 'job_date': '2023-05-15'},
    {'job_title': 'Data Analyst', 'job_skills': "['SQL', 'R', 'Tableau']", 'job_date': '2023-05-10'},
    {'job_title': 'Business Intelligence Developer', 'job_skills': "['SQL', 'PowerBI', 'Data Warehousing']", 'job_date': '2023-05-08'},
    {'job_title': 'Data Engineer', 'job_skills': "['Python', 'Spark', 'Hadoop']", 'job_date': '2023-05-18'},
    {'job_title': 'AI Specialist', 'job_skills': "['Python', 'PyTorch', 'AI Ethics']", 'job_date': '2023-05-20'}
]

for job in data_science_jobs:
    job['job_date'] = datetime.strptime(job['job_date'], '%Y-%m-%d')
    job['job_skills'] = ast.literal_eval(job['job_skills'])

data_science_jobs


[{'job_title': 'Data Scientist',
  'job_skills': ['Python', 'SQL', 'Machine Learning'],
  'job_date': '2023-05-12'},
 {'job_title': 'Machine Learning Engineer',
  'job_skills': ['Python', 'TensorFlow', 'Deep Learning'],
  'job_date': '2023-05-15'},
 {'job_title': 'Data Analyst',
  'job_skills': ['SQL', 'R', 'Tableau'],
  'job_date': '2023-05-10'},
 {'job_title': 'Business Intelligence Developer',
  'job_skills': ['SQL', 'PowerBI', 'Data Warehousing'],
  'job_date': '2023-05-08'},
 {'job_title': 'Data Engineer',
  'job_skills': ['Python', 'Spark', 'Hadoop'],
  'job_date': '2023-05-18'},
 {'job_title': 'AI Specialist',
  'job_skills': ['Python', 'PyTorch', 'AI Ethics'],
  'job_date': '2023-05-20'}]

### Classes

In [25]:
class BaseSalary:
  def __init__(self, base_salary, bonus_rate=0.1, symbol="$"):
    self.base_salary = base_salary
    self.bonus_rate = bonus_rate
    self.symbol = symbol
    self.total_salary = base_salary * (1 + bonus_rate)
    self.bonus = self.total_salary - self.base_salary

  def __repr__(self):
    return f'{self.symbol}{self.base_salary:,.0f}'

  def show_salary(self):
    return f'{self.symbol}{self.total_salary:,.0f}'

  def show_bonus(self):
    return f'{self.symbol}{self.bonus:,.0f}'

In [31]:
class JobPosting:
    def __init__(self, title, company, location, salary):
        self.title = title
        self.company = company
        self.location = location
        self.salary = salary
    
    def to_dict(self):
        return {
            'title': self.title,
            'company': self.company,
            'location': self.location,
            'salary': self.salary
        }

    def compare_salary(self, other_job):
        if not isinstance(other_job, JobPosting):
            raise TypeError("job must be a JobPosting (or subclass)")

        if type(other_job) is not type(self):
            raise TypeError("Can't compare different JobPosting subclasses")

        diff = f"Difference: {abs(self.salary - other_job.salary):,}"

        if self.salary > other_job.salary:
            return f"Winner: {self.title}. {diff}"
        else:
            return f"Winner: {other_job.title}. {diff}"
    
job = JobPosting('Data Scientist', 'Tech Innovations', 'New York', 120000)

print(job.to_dict())

job1 = JobPosting('Data Scientist', 'Tech Innovations', 'New York', 120000)
job2 = JobPosting('Data Analyst', 'Data Driven Co', 'San Francisco', 100000)

job2.compare_salary(job1)


{'title': 'Data Scientist', 'company': 'Tech Innovations', 'location': 'New York', 'salary': 120000}


'Winner: Data Scientist. Difference: 20,000'

### NumPy Intro

In [32]:
import random
import statistics
import numpy as np

salary_list = [random.randint(50000, 100000) for _ in range (10_000_000)]

In [33]:
%%timeit
statistics.mean(salary_list)

765 ms ± 15.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [34]:
%%timeit
np.mean(salary_list)

168 ms ± 1.41 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [42]:
# Job titles
job_titles = np.array(['Data Analyst', 'Data Scientist', 'Data Engineer', 'Machine Learning Engineer', 'AI Engineer'])

# Base salaries
base_salaries = np.array([60000, 80000, 75000, 90000, np.nan])

# np.nan - non numerical value

# Bonus rates
bonus_rates = np.array([.05, .1, .08, .12, np.nan])

In [43]:
total_salaries = base_salaries * (1 + bonus_rates)

total_salaries

array([ 63000.,  88000.,  81000., 100800.,     nan])

In [44]:
total_salaries.mean()

np.float64(nan)

In [46]:
np.nanmean(total_salaries)

np.float64(83200.0)